# Visualisasi Evaluasi Dialek Q03

Notebook ini membuat grafik **Score Decay & Ground Truth Relevance Mapping** untuk evaluasi dialek Q03. Format grafik dibuat konsisten dengan evaluasi chunking dan negasi:

- X = `rank_after_eval_filter`
- Y = `score`
- garis = kasus database dan kueri
- warna marker = nilai `relevance` dari ground truth

Data dibaca dari `evaluation/dialect_eval_results/r1` dan `evaluation/dialect_eval_results/r2`.

In [ ]:
from pathlib import Path
import os
import textwrap

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "legend.fontsize": 8,
})

DETAIL_CSV = "dialect_q03_detail_ndcg_at_30.csv"
SUMMARY_CSV = "dialect_q03_summary_ndcg_at_30.csv"

# Bisa dijalankan dari root repo atau dari folder evaluation/dialect_eval_results.
candidate_dirs = [Path("evaluation/dialect_eval_results"), Path(".")]
DATA_DIR = next((p for p in candidate_dirs if (p / "r1" / DETAIL_CSV).exists() and (p / "r2" / DETAIL_CSV).exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Tidak menemukan CSV evaluasi dialek pada r1 dan r2.")

FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

run_meta = {
    "r1": {
        "db_label": "DB US (Norm)",
        "collection_type": "normalized_us",
    },
    "r2": {
        "db_label": "DB UK (Raw)",
        "collection_type": "raw_uk",
    },
}
scenario_labels = {
    "dialect_off": "Query UK",
    "dialect_on": "Query US (Norm)",
}

detail_frames = []
summary_frames = []
for run_name, meta in run_meta.items():
    run_dir = DATA_DIR / run_name
    detail_part = pd.read_csv(run_dir / DETAIL_CSV)
    summary_part = pd.read_csv(run_dir / SUMMARY_CSV)
    for df in [detail_part, summary_part]:
        df["run"] = run_name
        df["db_label"] = meta["db_label"]
        df["collection_type"] = meta["collection_type"]
        df["query_label"] = df["scenario"].map(scenario_labels).fillna(df["scenario"])
        df["case_label"] = df["db_label"] + " + " + df["query_label"]
    detail_frames.append(detail_part)
    summary_frames.append(summary_part)

detail = pd.concat(detail_frames, ignore_index=True)
summary = pd.concat(summary_frames, ignore_index=True)

numeric_cols = [
    "rank_after_eval_filter", "rank_before_eval_filter", "score", "relevance",
    "chunk_index", "total_chunks", "chunk_tokens", "gt_rta_id",
    "m", "ef_construct", "search_hnsw_ef", "raw_qdrant_limit",
    "gt_rows", "raw_hit_count", "unique_article_count", "evaluated_rows",
    "matched_gt_rows_at_k", "dcg_at_30", "idcg_at_30", "ndcg_at_30", "elapsed_ms",
]
for df in [detail, summary]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

detail = detail.sort_values(["run", "scenario", "rank_after_eval_filter"]).reset_index(drop=True)
summary = summary.sort_values(["run", "scenario"]).reset_index(drop=True)
detail["relevance_label"] = detail["relevance"].fillna(0).astype(int).astype(str)
detail["doc_label"] = (
    detail["agreement"].astype(str)
    + " | " + detail["article_number"].astype(str)
    + " | " + detail["article_name"].astype(str)
)
detail["doc_short"] = detail["doc_label"].map(lambda s: textwrap.shorten(s, width=72, placeholder="..."))

print(f"Folder data: {DATA_DIR.resolve()}")
print(f"Detail rows: {len(detail)}")
print(f"Figure output: {FIG_DIR.resolve()}")


## Ringkasan Evaluasi

Tabel ini menampilkan ringkasan empat kasus pada matriks 2x2 dialek. Nilai NDCG@30 dibaca sebagai GT-filtered NDCG@30 sesuai metode evaluasi yang digunakan.

In [ ]:
summary[[
    "db_label", "query_label", "case_label", "final_query", "gt_rows",
    "raw_hit_count", "unique_article_count", "evaluated_rows",
    "matched_gt_rows_at_k", "ndcg_at_30", "elapsed_ms",
]]


## Grafik 1 - Score Decay & Relevance Mapping

Grafik ini menampilkan penurunan skor kemiripan vektor terhadap ranking hasil evaluasi. Panel kiri menunjukkan koleksi yang dinormalisasi ke ejaan US, sedangkan panel kanan menunjukkan koleksi raw dengan ejaan UK. Warna marker menunjukkan nilai relevance dari ground truth.

In [ ]:
score_decay = detail.copy().sort_values(["db_label", "query_label", "rank_after_eval_filter"])

line_colors = {
    "Query UK": "#2563eb",
    "Query US (Norm)": "#16a34a",
}
relevance_colors = {
    "0": "#dc2626",
    "1": "#f97316",
    "2": "#eab308",
    "3": "#15803d",
}
relevance_sizes = {
    "0": 50,
    "1": 70,
    "2": 95,
    "3": 125,
}

db_order = ["DB US (Norm)", "DB UK (Raw)"]
fig, axes = plt.subplots(1, len(db_order), figsize=(12, 5.8), sharey=True)
if len(db_order) == 1:
    axes = [axes]

for ax, db_label in zip(axes, db_order):
    db_df = score_decay[score_decay["db_label"] == db_label]
    for query_label, query_df in db_df.groupby("query_label"):
        query_df = query_df.sort_values("rank_after_eval_filter")
        ax.plot(
            query_df["rank_after_eval_filter"],
            query_df["score"],
            color=line_colors.get(query_label, "#64748b"),
            linewidth=2,
            alpha=0.72,
            label=f"Score decay - {query_label}",
            zorder=1,
        )
        for rel_label, rel_df in query_df.groupby("relevance_label"):
            ax.scatter(
                rel_df["rank_after_eval_filter"],
                rel_df["score"],
                s=relevance_sizes.get(rel_label, 70),
                color=relevance_colors.get(rel_label, "#64748b"),
                edgecolor="white",
                linewidth=0.9,
                alpha=0.94,
                zorder=3,
            )

    # Anotasi angka relevance dibuat kecil agar grafik tetap terbaca.
    for _, row in db_df.iterrows():
        ax.annotate(
            str(int(row["relevance"])),
            xy=(row["rank_after_eval_filter"], row["score"]),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=6.5,
            color="#111827",
        )

    ax.set_title(db_label)
    ax.set_xlabel("Rank setelah filter evaluasi")
    max_eval_rank = int(db_df["rank_after_eval_filter"].max())
    xticks = list(range(1, max_eval_rank + 1, 2))
    if xticks[-1] != max_eval_rank:
        xticks.append(max_eval_rank)
    ax.set_xticks(xticks)
    ax.grid(True, axis="y", alpha=0.35)

axes[0].set_ylabel("Vector similarity score")

line_handles = [
    plt.Line2D([0], [0], color=color, linewidth=2.2, label=f"Score decay - {label}")
    for label, color in line_colors.items()
]
relevance_handles = [
    plt.Line2D([0], [0], marker="o", color="none", label=f"Relevance {label}",
               markerfacecolor=color, markeredgecolor="white", markersize=8)
    for label, color in relevance_colors.items() if label in score_decay["relevance_label"].unique()
]

axes[-1].legend(handles=line_handles + relevance_handles, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=True)
fig.suptitle("Score Decay & Ground Truth Relevance Mapping", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 0.82, 1])

score_decay_path = FIG_DIR / "dialect_q03_score_decay_relevance_mapping.png"
fig.savefig(score_decay_path, bbox_inches="tight")
plt.show()

score_decay_path


## Grafik 2 - Score Decay per Database

Dua grafik berikut memisahkan visualisasi berdasarkan jenis database agar lebih mudah dibaca saat dimasukkan ke laporan. Format visual tetap sama: garis menunjukkan query, warna marker menunjukkan nilai relevance.

In [ ]:
def plot_single_dialect_score_decay(db_label: str, output_name: str):
    db_df = score_decay[score_decay["db_label"] == db_label].copy()
    fig, ax = plt.subplots(figsize=(9, 5.8))

    for query_label, query_df in db_df.groupby("query_label"):
        query_df = query_df.sort_values("rank_after_eval_filter")
        ax.plot(
            query_df["rank_after_eval_filter"],
            query_df["score"],
            color=line_colors.get(query_label, "#64748b"),
            linewidth=2,
            alpha=0.72,
            label=f"Score decay - {query_label}",
            zorder=1,
        )
        for rel_label, rel_df in query_df.groupby("relevance_label"):
            ax.scatter(
                rel_df["rank_after_eval_filter"],
                rel_df["score"],
                s=relevance_sizes.get(rel_label, 70),
                color=relevance_colors.get(rel_label, "#64748b"),
                edgecolor="white",
                linewidth=0.9,
                alpha=0.94,
                zorder=3,
            )

    for _, row in db_df.iterrows():
        ax.annotate(
            str(int(row["relevance"])),
            xy=(row["rank_after_eval_filter"], row["score"]),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#111827",
        )

    max_eval_rank = int(db_df["rank_after_eval_filter"].max())
    xticks = list(range(1, max_eval_rank + 1, 2))
    if xticks[-1] != max_eval_rank:
        xticks.append(max_eval_rank)
    ax.set_xticks(xticks)
    ax.set_xlabel("Rank setelah filter evaluasi")
    ax.set_ylabel("Vector similarity score")
    ax.set_title(f"Score Decay & Ground Truth Relevance Mapping - {db_label}")
    ax.grid(True, axis="y", alpha=0.35)

    line_handles = [
        plt.Line2D([0], [0], color=color, linewidth=2.2, label=f"Score decay - {label}")
        for label, color in line_colors.items()
    ]
    relevance_handles = [
        plt.Line2D([0], [0], marker="o", color="none", label=f"Relevance {label}",
                   markerfacecolor=color, markeredgecolor="white", markersize=8)
        for label, color in relevance_colors.items() if label in db_df["relevance_label"].unique()
    ]
    ax.legend(handles=line_handles + relevance_handles, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=True)
    plt.tight_layout(rect=[0, 0, 0.78, 1])

    output_path = FIG_DIR / output_name
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return output_path

db_us_path = plot_single_dialect_score_decay("DB US (Norm)", "dialect_q03_score_decay_db_us_norm.png")
db_uk_path = plot_single_dialect_score_decay("DB UK (Raw)", "dialect_q03_score_decay_db_uk_raw.png")

db_us_path, db_uk_path


## Tabel Ringkas Score Decay

Tabel ini membantu membaca grafik dengan membandingkan skor awal, skor akhir, rata-rata skor, dan total relevance pada setiap kasus.

In [ ]:
score_decay_summary = (
    score_decay.groupby(["db_label", "query_label", "case_label"], as_index=False)
    .agg(
        evaluated_rows=("rank_after_eval_filter", "count"),
        first_score=("score", "first"),
        last_score=("score", "last"),
        mean_score=("score", "mean"),
        relevance_sum=("relevance", "sum"),
        mean_relevance=("relevance", "mean"),
    )
)
score_decay_summary


## Catatan Interpretasi

Grafik ini dipakai untuk membandingkan pola penurunan skor dan pemetaan relevance antar kondisi dialek. Karena evaluasi menggunakan ground truth pool filtering, ranking pada sumbu X adalah urutan artikel yang cocok dengan ground truth pool, bukan seluruh top-k kandidat mentah dari Qdrant.